# 01. Сбор данных

**ЛР 2, шаг 6.** Ноутбук загружает пространственные данные из OpenStreetMap и сохраняет их в `data/raw/`.

Код написан для учебного примера **AquaMap** (точки питьевой воды в Екатеринбурге). Для своего варианта поменяйте параметры в следующей ячейке: территорию и теги объектов. Подобрать теги можно на [wiki.openstreetmap.org](https://wiki.openstreetmap.org/wiki/Map_features) и проверить в [Overpass Turbo](https://overpass-turbo.eu).

Данные OpenStreetMap распространяются по лицензии ODbL. На картах и в отчётах указывайте: «© участники OpenStreetMap».\n\n[![Открыть в Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/OWNER/REPO/blob/main/notebooks/01_data_collection.ipynb)\n\nЧтобы кнопка открывала ваш ноутбук, замените в ссылке `OWNER/REPO` на владельца и имя своего репозитория (например, `ivan-p/pm-lab-00-aquamap`). Другие способы открыть ноутбук описаны в приложении Б.

## Настройка среды

Ноутбук рассчитан на **Google Colab** (как получить бесплатный доступ: приложение Б к заданию ЛР 2). Он работает и локально, в Jupyter или VS Code.

Что делает ячейка ниже:
1. В Colab ставит недостающие библиотеки и подключает Google Диск. Colab попросит разрешение на доступ к Диску: выберите тот же аккаунт Google, под которым открыт Colab.
2. Находит рабочую папку группы на Диске. Все данные и результаты сохраняются туда, поэтому после отключения сессии ничего не пропадёт.
3. Создаёт внутри неё ту же структуру папок, что и в репозитории: `data/raw`, `data/manual`, `data/processed`, `docs/img`, `docs/maps`.

**Перед первым запуском** впишите в `PROJECT_FOLDER` имя папки группы на Google Диске.

Запуск всего ноутбука: **Среда выполнения → Выполнить все** (Runtime → Run all).

In [ ]:
# === Настройка среды: запустите эту ячейку первой ===
import sys, subprocess
from pathlib import Path

# Имя папки группы в «Моём диске» Google. Обычно совпадает с именем репозитория.
PROJECT_FOLDER = "pm-lab-8-ForestFireMonitoring"

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", 'geopandas>=1.0', 'osmnx>=2.0', 'mapclassify'], check=True)
    from google.colab import drive
    drive.mount("/content/drive")
    BASE = Path("/content/drive/MyDrive") / PROJECT_FOLDER
    if not BASE.exists():
        raise FileNotFoundError(
            f"На Google Диске нет папки «{PROJECT_FOLDER}». Создайте её или добавьте ярлык "
            "общей папки группы в «Мой диск» (приложение Б, шаг 2) и запустите ячейку снова.")
else:
    BASE = Path("..")   # локальный запуск из папки notebooks/ репозитория

RAW, MANUAL, PROCESSED = BASE / "data" / "raw", BASE / "data" / "manual", BASE / "data" / "processed"
IMG, MAPS = BASE / "docs" / "img", BASE / "docs" / "maps"
for folder in (RAW, MANUAL, PROCESSED, IMG, MAPS):
    folder.mkdir(parents=True, exist_ok=True)
print("Среда:", "Google Colab" if IN_COLAB else "локальная", "· рабочая папка:", BASE.resolve())

In [ ]:
import osmnx as ox

# === Параметры вашего варианта ===
PLACE = "Республика Башкортостан, Россия"
DISTRICT_ADMIN_LEVEL = "9"

# Слои: имя файла -> теги OSM. Адаптировано под мониторинг пожаров.
LAYERS = {
    "forests": {"landuse": ["forest"], "natural": ["wood"]},
    "cameras": {"man_made": ["surveillance"]},
    "settlements": {"place": ["city", "town", "village"]},
}

# Атрибуты, которые сохраняем
KEEP_COLUMNS = ["name", "landuse", "natural", "man_made", "surveillance:type",
                "place", "population", "operator", "ref"]

# Настройки OSMnx
ox.settings.use_cache = True
ox.settings.cache_folder = str(RAW / "cache")
ox.settings.requests_timeout = 300
ox.settings.log_console = False

In [ ]:
import geopandas as gpd
import pandas as pd
from datetime import datetime, timezone

def to_layer(gdf: gpd.GeoDataFrame) -> gpd.GeoDataFrame:
    """Приводит результат osmnx к простой таблице: тип и id объекта OSM, нужные атрибуты, геометрия."""
    gdf = gdf.reset_index()
    gdf = gdf.rename(columns={"element": "osm_type", "id": "osm_id",
                              "element_type": "osm_type", "osmid": "osm_id"})
    cols = ["osm_type", "osm_id"] + [c for c in KEEP_COLUMNS if c in gdf.columns] + ["geometry"]
    gdf = gdf[cols].copy()
    for c in gdf.columns:
        if c != "geometry" and gdf[c].dtype == object:
            gdf[c] = gdf[c].map(lambda v: v if v is None or isinstance(v, str) else str(v))
    return gdf.set_crs(4326, allow_override=True)

log = {"place": PLACE, "downloaded_at": datetime.now(timezone.utc).isoformat(timespec="seconds"),
       "source": "OpenStreetMap via Overpass API (osmnx)", "license": "ODbL 1.0", "layers": {}}

## Граница территории и районы

In [ ]:
# === НАСТРОЙКА СЕРВЕРА ===
# Пробуем публичный сервер OSMnx по умолчанию (он может быть перегружен, но попробуем)
ox.settings.requests_timeout = 600
ox.settings.log_console = True

PLACE = "Уфа, Республика Башкортостан, Россия"
all_districts = []

print(f"Пробуем загрузить данные для: {PLACE}...")

# Делаем 3 попытки с паузой
for attempt in range(3):
    try:
        # Загружаем административные границы
        admin = ox.features_from_place(PLACE, tags={"boundary": "administrative"})

        # Фильтруем по уровню (9 для районов города)
        if 'admin_level' in admin.columns:
            admin = admin[admin.get('admin_level') == "9"]

        # Оставляем только полигоны
        admin = admin[admin.geometry.geom_type.isin(['Polygon', 'MultiPolygon'])]

        if not admin.empty:
            clean_districts = to_layer(admin)
            all_districts.append(clean_districts)
            print(f"  -> УСПЕХ! Загружено районов: {len(clean_districts)}")
            break # Если получилось, выходим из цикла попыток
        else:
            print(f"  -> Районы с уровнем 9 не найдены. Пробуем еще раз...")

    except Exception as e:
        print(f"  -> Ошибка (попытка {attempt+1}): {e}")
        print("     Ждем 15 секунд перед следующей попыткой...")
        time.sleep(15)

# === ОБЪЕДИНЕНИЕ РЕЗУЛЬТАТОВ ===
if all_districts:
    districts = pd.concat(all_districts, ignore_index=True)
    districts = gpd.GeoDataFrame(districts, geometry='geometry', crs="EPSG:4326")

    # Сохраняем результат
    districts.to_file(RAW / "districts.gpkg", driver="GPKG")

    log["layers"]["districts"] = len(districts)
    print(f"\nВСЕГО загружено районов: {len(districts)}")
    display(districts[['name']].head(10))
else:
    print("\nНе удалось загрузить Уфу. Сервер OSM полностью лежит. Придется качать готовый файл с Geofabrik.")

## Тематические слои

Загрузка крупного города может занять несколько минут. Если сервер Overpass не отвечает (ошибка 429 или 504), подождите и запустите ячейку ещё раз: уже загруженное возьмётся из кэша.

In [ ]:
for name, tags in LAYERS.items():
    gdf = to_layer(ox.features_from_place(PLACE, tags=tags))
    gdf.to_file(RAW / f"{name}.gpkg", driver="GPKG")
    log["layers"][name] = len(gdf)
    print(f"{name}: {len(gdf)} объектов, типы геометрий: {gdf.geom_type.value_counts().to_dict()}")

(RAW / "_download_log.json").write_text(json.dumps(log, ensure_ascii=False, indent=2), encoding="utf-8")
log

## Быстрая проверка на карте

In [ ]:
water = gpd.read_file(RAW / "water_points.gpkg")
m = boundary.explore(color="gray", style_kwds={"fillOpacity": 0.05})
water.explore(m=m, color="blue", tooltip=["name", "amenity", "man_made"])
m

## Если Overpass недоступен

1. Откройте [Overpass Turbo](https://overpass-turbo.eu), выполните запрос по своей территории и сохраните результат: **Export → GeoJSON**. Положите файл в папку `data/raw` (в Colab — на Google Диске: значок папки слева → `drive/MyDrive/<папка группы>/data/raw`) и прочитайте его через `gpd.read_file(RAW / "имя_файла.geojson")`.
2. Либо скачайте выгрузку региона с [Geofabrik](https://download.geofabrik.de) и откройте её в QGIS (слои points, multipolygons), отфильтруйте по тегам и сохраните в GeoPackage.

Каким бы способом вы ни получили данные, запишите его в `data/README.md`, чтобы другой участник группы мог повторить загрузку.

## Что записать в реестр источников

Для каждого слоя внесите или обновите карточку в `data/sources.md`: дата загрузки (из `_download_log.json`), число объектов, лицензия, система координат.